# Sales 2025 SKU Real vs Edit Visualization

Notebook ini hanya untuk visualisasi `Sales 2025`.

Tujuan:
- ambil kolom sales utama sesuai request
- bandingkan SKU versi `Real` dan `Edit`
- buat `Edit` dengan acuan SKU terbaru dari `Sales 2026` bila tersedia
- tampilkan line chart 2025 per SKU berbasis `CF`
- melihat apakah edit SKU membuat series lebih nyambung, misalnya ukuran lama 3.1L disetarakan ke ukuran baru 3L

In [ ]:
from pathlib import Path
import re
import warnings
import zipfile

import numpy as np
import pandas as pd

try:
    import plotly.express as px
    HAS_PLOTLY = True
except ModuleNotFoundError:
    HAS_PLOTLY = False
    px = None

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ModuleNotFoundError:
    HAS_MATPLOTLIB = False
    plt = None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

## 1. Source File

Di Colab, ubah `SALES_FILE` ke path Google Drive kamu kalau perlu.

In [ ]:
LOCAL_PRIMARY = Path(r"C:\Users\didik.priyo.id\OneDrive - PT AJEINDONESIA\Commercial_Analyst\Data Sales\Indonesia Sales Dashboard 2026.xlsm")
LOCAL_SNAPSHOT = Path(r"C:\Users\didik.priyo.id\Documents\Codex\2026-08-19\c-users-didik-priyo-id-onedrive\work\Indonesia Sales Dashboard 2026 PT snapshot.xlsm")
LOCAL_FALLBACK = Path(r"C:\Users\didik.priyo.id\OneDrive - Ajethai\INDONESIA\BP INDONESIA\KPI Daily Sales\Indonesia Sales Dashboard 2026.xlsm")
COLAB_FILE = Path("/content/drive/MyDrive/Data Sales/Indonesia Sales Dashboard 2026.xlsm")

if Path("/content").exists() and not COLAB_FILE.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print("Google Drive mount skipped:", exc)

def readable_excel(path):
    try:
        with zipfile.ZipFile(path) as zf:
            return "[Content_Types].xml" in zf.namelist()
    except Exception:
        return False

for candidate in [COLAB_FILE, LOCAL_PRIMARY, LOCAL_SNAPSHOT, LOCAL_FALLBACK]:
    if candidate.exists() and readable_excel(candidate):
        SALES_FILE = candidate
        break
else:
    SALES_FILE = LOCAL_PRIMARY

print("Using:", SALES_FILE)

## 2. Load Sales 2025

In [ ]:
raw = pd.read_excel(SALES_FILE, sheet_name="Sales 2025", engine="openpyxl")
try:
    raw_2026_reference = pd.read_excel(SALES_FILE, sheet_name="Sales 2026", engine="openpyxl")
except Exception as exc:
    raw_2026_reference = None
    print("Sales 2026 reference skipped:", exc)

print("Sales 2025 shape:", raw.shape)
if raw_2026_reference is not None:
    print("Sales 2026 reference shape:", raw_2026_reference.shape)
display(raw.head(3))
print("Columns:")
for i, col in enumerate(raw.columns, start=1):
    print(i, repr(col))

## 3. Build Real vs Edit View

`Real` memakai kolom asli jika tersedia:
- `Item Code (Real)`
- `Short Item Description (Edit).1` atau `Short Item Description`

`Edit` dibuat ulang dari rule alignment:
- ukuran `1625 / 1.625` disamakan ke kelompok `1000 / 1`
- ukuran `3100 / 3.1` disamakan ke kelompok `3000 / 3`
- ukuran `400 / 0.4` dengan flavour yang sama disamakan walau SKU beda
- acuan `Item Code Edit` dan `Short Item Description Edit` diambil dari SKU terbaru di `Sales 2026` jika tersedia
- jika tidak ada di `Sales 2026`, fallback ke row terbaru di `Sales 2025`

In [ ]:
def first_existing(df, names, required=True):
    for name in names:
        if name in df.columns:
            return name
    if required:
        raise KeyError(f"None of these columns were found: {names}")
    return None

date_col = first_existing(raw, ["Date"])
channel_group_col = first_existing(raw, ["Channel Group"])
branch_col = first_existing(raw, ["Branch"])
channel_col = first_existing(raw, ["Channel"])
cust_code_col = first_existing(raw, ["Cust Code"])
customer_name_col = first_existing(raw, ["Customer Name"])
format_col = first_existing(raw, ["Format"])
brand_col = first_existing(raw, ["Brand"], required=False)
flavor_col = first_existing(raw, ["Flavor"], required=False)
box_content_col = first_existing(raw, ["Box Content"])
cf_col = first_existing(raw, ["CF"])

real_desc_col = first_existing(raw, ["Short Item Description (Real)", "Short Item Description (Edit).1", "Short Item Description"], required=False)
real_code_col = first_existing(raw, ["Item Code (Real)", "Item Code"], required=False)

view = pd.DataFrame()
view["Date"] = pd.to_datetime(raw[date_col], errors="coerce")
view["Channel Group"] = raw[channel_group_col]
view["Branch"] = raw[branch_col]
view["Channel"] = raw[channel_col]
view["Cust Code"] = raw[cust_code_col].astype("string")
view["Customer Name"] = raw[customer_name_col]
view["Brand"] = raw[brand_col] if brand_col else ""
view["Flavor"] = raw[flavor_col] if flavor_col else ""
view["Format"] = pd.to_numeric(raw[format_col], errors="coerce")
view["Box Content"] = raw[box_content_col].astype("string")
view["CF"] = pd.to_numeric(raw[cf_col], errors="coerce").fillna(0)
view["Item Code Real"] = raw[real_code_col].astype("string")
view["Short Item Description Real"] = raw[real_desc_col].astype("string")

view = view.dropna(subset=["Date"]).copy()
view["Month"] = view["Date"].dt.to_period("M").dt.to_timestamp()

def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().upper()
    x = re.sub(r"\s+", " ", x)
    return x

def clean_code(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    return x[:-2] if x.endswith(".0") else x

def align_size(fmt):
    if pd.isna(fmt):
        return ""
    fmt = float(fmt)
    if np.isclose(fmt, 1.625):
        return "1"
    if np.isclose(fmt, 3.1):
        return "3"
    if np.isclose(fmt, 0.4):
        return "0.4"
    return f"{fmt:g}"

view["Brand Norm"] = view["Brand"].map(clean_text)
view["Flavor Norm"] = view["Flavor"].map(clean_text)
view["Alignment Size"] = view["Format"].map(align_size)
view["Item Code Real"] = view["Item Code Real"].map(clean_code)
view["Short Item Description Real"] = view["Short Item Description Real"].map(clean_text)
view["SKU Real"] = view["Item Code Real"].map(clean_code) + " | " + view["Short Item Description Real"].map(clean_text)

reference_sort = view.sort_values(
    ["Brand Norm", "Flavor Norm", "Alignment Size", "Date", "CF"],
    ascending=[True, True, True, False, False],
)
alignment_reference_2025 = (
    reference_sort
    .drop_duplicates(["Brand Norm", "Flavor Norm", "Alignment Size"])
    [[
        "Brand Norm", "Flavor Norm", "Alignment Size", "Date",
        "Item Code Real", "Short Item Description Real", "Format", "Box Content",
    ]]
    .rename(
        columns={
            "Date": "Reference Date",
            "Item Code Real": "Item Code Edit",
            "Short Item Description Real": "Short Item Description Edit",
            "Format": "Reference Format",
            "Box Content": "Reference Box Content",
        }
    )
)

source_summary = (
    view.groupby(["Brand Norm", "Flavor Norm", "Alignment Size"], as_index=False)
    .agg(
        source_formats=("Format", lambda s: ", ".join(map(str, sorted(set(s.dropna()))))),
        source_item_codes=("Item Code Real", lambda s: ", ".join(sorted(set(filter(None, s))))),
        source_descriptions=("Short Item Description Real", lambda s: " || ".join(sorted(set(filter(None, s)))[:8])),
        total_cf=("CF", "sum"),
        row_count=("CF", "size"),
    )
)
alignment_reference = alignment_reference_2025.merge(
    source_summary,
    on=["Brand Norm", "Flavor Norm", "Alignment Size"],
    how="left",
)

if raw_2026_reference is not None:
    ref26 = pd.DataFrame()
    ref26["Reference Date 2026"] = pd.to_datetime(raw_2026_reference["fecha_liquidacion"], errors="coerce")
    ref26["Brand Norm"] = raw_2026_reference["desc_marca"].map(clean_text)
    ref26["Flavor Norm"] = raw_2026_reference["desc_sabor"].map(clean_text)
    ref26["Reference Format 2026"] = pd.to_numeric(raw_2026_reference["desc_formato"], errors="coerce")
    ref26["Alignment Size"] = ref26["Reference Format 2026"].map(align_size)
    ref26["Item Code Edit 2026"] = raw_2026_reference["cod_articulo"].map(clean_code)
    ref26["Short Item Description Edit 2026"] = raw_2026_reference["desc_articulo_corto"].map(clean_text)
    ref26["Reference Box Content 2026"] = raw_2026_reference["cant_contenido"].astype("string")
    ref26["CF 2026"] = pd.to_numeric(raw_2026_reference["CF"], errors="coerce").fillna(0)
    ref26 = ref26.dropna(subset=["Reference Date 2026"])
    ref26 = (
        ref26.sort_values(
            ["Brand Norm", "Flavor Norm", "Alignment Size", "Reference Date 2026", "CF 2026"],
            ascending=[True, True, True, False, False],
        )
        .drop_duplicates(["Brand Norm", "Flavor Norm", "Alignment Size"])
    )
    alignment_reference = alignment_reference.merge(
        ref26[
            [
                "Brand Norm", "Flavor Norm", "Alignment Size",
                "Reference Date 2026", "Item Code Edit 2026",
                "Short Item Description Edit 2026", "Reference Format 2026",
                "Reference Box Content 2026",
            ]
        ],
        on=["Brand Norm", "Flavor Norm", "Alignment Size"],
        how="left",
    )
    has_2026 = alignment_reference["Item Code Edit 2026"].notna() & alignment_reference["Item Code Edit 2026"].ne("")
    alignment_reference.loc[has_2026, "Reference Date"] = alignment_reference.loc[has_2026, "Reference Date 2026"]
    alignment_reference.loc[has_2026, "Item Code Edit"] = alignment_reference.loc[has_2026, "Item Code Edit 2026"]
    alignment_reference.loc[has_2026, "Short Item Description Edit"] = alignment_reference.loc[has_2026, "Short Item Description Edit 2026"]
    alignment_reference.loc[has_2026, "Reference Format"] = alignment_reference.loc[has_2026, "Reference Format 2026"]
    alignment_reference.loc[has_2026, "Reference Box Content"] = alignment_reference.loc[has_2026, "Reference Box Content 2026"]
    alignment_reference["Reference Source"] = np.where(has_2026, "Sales 2026 latest", "Sales 2025 latest")
else:
    alignment_reference["Reference Source"] = "Sales 2025 latest"

view = view.merge(
    alignment_reference[
        [
            "Brand Norm", "Flavor Norm", "Alignment Size",
            "Item Code Edit", "Short Item Description Edit",
            "Reference Date", "Reference Format", "Reference Box Content",
        ]
    ],
    on=["Brand Norm", "Flavor Norm", "Alignment Size"],
    how="left",
)
view["SKU Edit"] = view["Item Code Edit"].map(clean_code) + " | " + view["Short Item Description Edit"].map(clean_text)
view["Real != Edit"] = view["SKU Real"] != view["SKU Edit"]

selected_columns = [
    "Date", "Channel Group", "Branch", "Channel", "Cust Code", "Customer Name",
    "Short Item Description Edit", "Item Code Edit", "Format", "Alignment Size", "Box Content", "CF",
    "Short Item Description Real", "Item Code Real", "SKU Real", "SKU Edit", "Real != Edit",
    "Reference Date", "Reference Format", "Reference Box Content",
]
sales_2025_visual = view[selected_columns].copy()

print("Rows:", len(sales_2025_visual))
print("Rows changed by edit:", int(sales_2025_visual["Real != Edit"].sum()))
display(sales_2025_visual.head(10))
display(
    alignment_reference.sort_values("total_cf", ascending=False)
    [[
        "Brand Norm", "Flavor Norm", "Alignment Size", "Reference Date",
        "Item Code Edit", "Short Item Description Edit", "Reference Format",
        "Reference Source", "source_formats", "source_item_codes", "total_cf", "row_count",
    ]]
    .head(30)
)

## 4. Monthly CF: Real vs Edit

Chart pertama membandingkan jumlah SKU distinct sebelum dan sesudah edit.

In [ ]:
distinct_monthly = (
    view.groupby("Month")
    .agg(
        real_sku_count=("SKU Real", "nunique"),
        edit_sku_count=("SKU Edit", "nunique"),
        cf=("CF", "sum"),
    )
    .reset_index()
)

display(distinct_monthly)

if HAS_PLOTLY:
    fig = px.line(
        distinct_monthly,
        x="Month",
        y=["real_sku_count", "edit_sku_count"],
        markers=True,
        title="Distinct SKU Count per Month: Real vs Edit",
    )
    fig.show()
elif HAS_MATPLOTLIB:
    ax = distinct_monthly.plot(x="Month", y=["real_sku_count", "edit_sku_count"], marker="o", figsize=(12, 5))
    ax.set_title("Distinct SKU Count per Month: Real vs Edit")
    ax.grid(True, alpha=0.3)
    plt.show()

## 5. Line Chart per SKU - Real

In [ ]:
real_monthly = (
    view.groupby(["Month", "SKU Real"], as_index=False)
    .agg(CF=("CF", "sum"))
)

top_real = (
    real_monthly.groupby("SKU Real")["CF"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .index
)

real_top = real_monthly[real_monthly["SKU Real"].isin(top_real)].copy()

if HAS_PLOTLY:
    fig = px.line(
        real_top,
        x="Month",
        y="CF",
        color="SKU Real",
        markers=True,
        title="Sales 2025 Monthly CF by SKU - Real Top 20",
    )
    fig.show()
elif HAS_MATPLOTLIB:
    pivot = real_top.pivot_table(index="Month", columns="SKU Real", values="CF", aggfunc="sum", fill_value=0)
    ax = pivot.plot(figsize=(14, 7), marker="o")
    ax.set_title("Sales 2025 Monthly CF by SKU - Real Top 20")
    ax.grid(True, alpha=0.3)
    plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.show()
else:
    display(real_top.head(50))

## 6. Line Chart per SKU - Edit

In [ ]:
edit_monthly = (
    view.groupby(["Month", "SKU Edit"], as_index=False)
    .agg(CF=("CF", "sum"))
)

top_edit = (
    edit_monthly.groupby("SKU Edit")["CF"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .index
)

edit_top = edit_monthly[edit_monthly["SKU Edit"].isin(top_edit)].copy()

if HAS_PLOTLY:
    fig = px.line(
        edit_top,
        x="Month",
        y="CF",
        color="SKU Edit",
        markers=True,
        title="Sales 2025 Monthly CF by SKU - Edit Top 20",
    )
    fig.show()
elif HAS_MATPLOTLIB:
    pivot = edit_top.pivot_table(index="Month", columns="SKU Edit", values="CF", aggfunc="sum", fill_value=0)
    ax = pivot.plot(figsize=(14, 7), marker="o")
    ax.set_title("Sales 2025 Monthly CF by SKU - Edit Top 20")
    ax.grid(True, alpha=0.3)
    plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.show()
else:
    display(edit_top.head(50))

## 7. Side-by-Side: SKU yang Berubah

Chart ini hanya menampilkan SKU yang terkena edit, sehingga efek penyamaan lebih kelihatan.

In [ ]:
changed = view[view["Real != Edit"]].copy()

changed_summary = (
    changed.groupby(["SKU Real", "SKU Edit"], as_index=False)
    .agg(
        rows=("CF", "size"),
        cf=("CF", "sum"),
        first_date=("Date", "min"),
        last_date=("Date", "max"),
    )
    .sort_values("cf", ascending=False)
)

display(changed_summary.head(50))

top_changed_edit = changed_summary.head(12)["SKU Edit"].unique()
changed_long = pd.concat(
    [
        changed.assign(SKU_View="Real", SKU=changed["SKU Real"]),
        changed.assign(SKU_View="Edit", SKU=changed["SKU Edit"]),
    ],
    ignore_index=True,
)
changed_long = changed_long[changed_long["SKU Edit"].isin(top_changed_edit)]
changed_monthly = (
    changed_long.groupby(["Month", "SKU_View", "SKU"], as_index=False)
    .agg(CF=("CF", "sum"))
)

if HAS_PLOTLY:
    fig = px.line(
        changed_monthly,
        x="Month",
        y="CF",
        color="SKU",
        line_dash="SKU_View",
        markers=True,
        title="Changed SKU Monthly CF: Real vs Edit",
    )
    fig.show()
elif HAS_MATPLOTLIB:
    for view_name, data in changed_monthly.groupby("SKU_View"):
        pivot = data.pivot_table(index="Month", columns="SKU", values="CF", aggfunc="sum", fill_value=0)
        ax = pivot.plot(figsize=(14, 6), marker="o")
        ax.set_title(f"Changed SKU Monthly CF - {view_name}")
        ax.grid(True, alpha=0.3)
        plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
        plt.tight_layout()
        plt.show()

## 8. Filter Manual

Ubah `keyword` untuk melihat SKU tertentu, contoh `LIME`, `3LT`, `3.1`, `NIPIS`.

In [ ]:
keyword = "LIME"

mask = (
    view["SKU Real"].str.contains(keyword, case=False, na=False)
    | view["SKU Edit"].str.contains(keyword, case=False, na=False)
)
filtered = view[mask].copy()

filtered_long = pd.concat(
    [
        filtered.assign(SKU_View="Real", SKU=filtered["SKU Real"]),
        filtered.assign(SKU_View="Edit", SKU=filtered["SKU Edit"]),
    ],
    ignore_index=True,
)
filtered_monthly = (
    filtered_long.groupby(["Month", "SKU_View", "SKU"], as_index=False)
    .agg(CF=("CF", "sum"))
)

display(filtered_monthly.head(50))

if HAS_PLOTLY:
    fig = px.line(
        filtered_monthly,
        x="Month",
        y="CF",
        color="SKU",
        line_dash="SKU_View",
        markers=True,
        title=f"Manual Filter: {keyword} - Real vs Edit",
    )
    fig.show()
elif HAS_MATPLOTLIB:
    for view_name, data in filtered_monthly.groupby("SKU_View"):
        pivot = data.pivot_table(index="Month", columns="SKU", values="CF", aggfunc="sum", fill_value=0)
        ax = pivot.plot(figsize=(14, 6), marker="o")
        ax.set_title(f"Manual Filter: {keyword} - {view_name}")
        ax.grid(True, alpha=0.3)
        plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
        plt.tight_layout()
        plt.show()